# Explore dataset structure

In [ ]:
from dirs import *

from xflow.utils.io import scan_files
from xflow.utils.sql import merge_sqlite_dbs
from xflow import SqlProvider, PyTorchPipeline
from xflow.data import build_transforms_from_config
from xflow.utils.visualization import plot_image
from xflow.extensions.physics.pipeline import CachedBasisPipeline, IndexCombinator
from xflow.extensions.physics import pattern_gen
from tqdm import tqdm
from copy import deepcopy

from config_utils import load_config, detect_machine
import pandas as pd
import sqlite3
from utils import *

experiment_name = "CLEAR_visualization"  
config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

# Load dataset

Different with training pipeline, data sample meta data is given

Mainly the run time sythetic dataset need to be compiled and generate here

In [ ]:
"""
Contract:
1. Unify the data format for visualization like (2, N, 1, 256, 256)
   (fiber_input_output_image_type, num_samples, C, H, W)
2. Add labels/colors or class to data samples with index based order matter.
3. Union data samples, labels and colors into three single iterable for downstream visualization.
"""

# ============================================================
# Small helpers
# ============================================================
def make_transforms(parent_dir, base_transform_cfg, drop_last=True):
    """
    Build transforms safely without mutating config["data"]["transforms"]["torch"].
    This fixes the repeated insert bug.
    """
    transform_cfg = [
        {
            "name": "add_parent_dir",
            "params": {"parent_dir": parent_dir},
        },
        *deepcopy(base_transform_cfg),
    ]
    transforms = build_transforms_from_config(transform_cfg)
    return transforms[:-1] if drop_last else transforms


def to_modal_array(x):
    """
    Normalize data to shape (2, N, C, H, W).

    Common cases:
    - list/tuple with length 2, each item (N, C, H, W)
    - ndarray already in shape (2, N, C, H, W)
    - ndarray in shape (N, 2, C, H, W)
    """
    if isinstance(x, np.ndarray):
        arr = x
    else:
        arr = np.stack(x, axis=0)

    if arr.ndim != 5:
        raise ValueError(f"Expected ndim=5, got shape={arr.shape}")

    if arr.shape[0] == 2:
        return arr

    if arr.shape[1] == 2:
        return np.transpose(arr, (1, 0, 2, 3, 4))

    raise ValueError(f"Cannot convert to (2, N, C, H, W), got shape={arr.shape}")


def build_sql_provider(db_path, sql, subsample_n=None, seed=None):
    provider = SqlProvider(
        sources={"connection": db_path, "sql": sql},
        output_config={"list": "image_path"},
    )
    if subsample_n is not None:
        provider = provider.subsample(n_samples=subsample_n, seed=seed)
    return provider


def load_sql_pipeline(db_path, sql, parent_dir, *, subsample_n=None):
    """
    Standard SQL -> PyTorchPipeline -> numpy -> (2, N, C, H, W)
    """
    provider = build_sql_provider(
        db_path=db_path,
        sql=sql,
        subsample_n=subsample_n,
        seed=config["seed"],
    )
    transforms = make_transforms(
        parent_dir=parent_dir,
        base_transform_cfg=config["data"]["transforms"]["torch"],
        drop_last=True,
    )
    pipeline = PyTorchPipeline(provider, transforms).to_numpy()
    return to_modal_array(pipeline)


def make_bundle(samples, label, color, name=None):
    """
    samples: (2, N, C, H, W)
    """
    n = samples.shape[1]
    return {
        "name": name,
        "samples": samples,
        "labels": [label] * n,
        "colors": [color] * n,
    }


def concat_bundles(*bundles, name="merged"):
    """
    Concatenate multiple bundles along sample axis.
    """
    samples = np.concatenate([b["samples"] for b in bundles], axis=1)
    labels = sum([b["labels"] for b in bundles], [])
    colors = sum([b["colors"] for b in bundles], [])
    return {
        "name": name,
        "samples": samples,
        "labels": labels,
        "colors": colors,
    }


def build_sgm_stream():
    """
    Reused by DMD synthetic builders.
    """
    canvas = pattern_gen.DynamicPatterns(*config["simulation"]["canvas_size"])
    canvas.set_postprocess_fns(
        build_transforms_from_config(config["simulation"]["process_functions"])
    )
    canvas._distributions = [
        pattern_gen.StaticGaussianDistribution(canvas)
        for _ in range(config["simulation"]["total_Guassian_num"])
    ]
    canvas.set_threshold(config["simulation"]["minimum_pixel_threshold"])

    stream = canvas.pattern_stream(
        std_1=config["simulation"]["std_1"],
        std_2=config["simulation"]["std_2"],
        max_intensity=config["simulation"]["max_intensity"],
        fade_rate=config["simulation"]["fade_rate"],
        distribution=config["simulation"]["distribution"],
    )
    return stream


def build_dmd_syth_plus_eval(
    db_path,
    extracted_dir,
    orth_sql,
    eval_sql,
    syth_label,
    eval_label,
    syth_color,
    eval_color,
    syth_take_n=1000,
):
    """
    DMD orthogonal basis + synthetic augmentation + eval set
    """
    # ---------- eval ----------
    eval_provider = build_sql_provider(db_path, eval_sql)
    eval_transforms = make_transforms(
        parent_dir=extracted_dir,
        base_transform_cfg=config["data"]["transforms"]["torch"],
        drop_last=True,
    )
    eval_pipeline = PyTorchPipeline(eval_provider, eval_transforms).to_numpy()
    eval_samples = to_modal_array(eval_pipeline)

    # ---------- synthetic ----------
    orth_provider = build_sql_provider(db_path, orth_sql)

    full_transforms = make_transforms(
        parent_dir=extracted_dir,
        base_transform_cfg=config["data"]["transforms"]["torch"],
        drop_last=False,
    )

    combinator = IndexCombinator(
        pattern_provider=build_sgm_stream(),
        transforms=build_transforms_from_config(config["combinator"]["transforms"]["torch"]),
    )

    train_dataset = CachedBasisPipeline(
        orth_provider,
        combinator=combinator,
        transforms=full_transforms,
        num_samples=config["data"]["total_train_samples"],
        seed=config["seed"],
        eager=True,
    )

    # Important fix:
    # do not call next(iter(train_dataset)) each time
    iterator = iter(train_dataset)

    samples = []
    for _ in tqdm(range(syth_take_n)):
        samples.append(next(iterator))

    samples = np.stack(samples, axis=0)              # (N, 2, C, H, W)
    samples = np.transpose(samples, (1, 0, 2, 3, 4))  # (2, N, C, H, W)

    # ---------- merge ----------
    bundle_syth = make_bundle(samples, syth_label, syth_color)
    bundle_eval = make_bundle(eval_samples, eval_label, eval_color)

    return concat_bundles(bundle_syth, bundle_eval)


# ============================================================
# Dataset builders
# Keep this flat and readable
# ============================================================
def build_dmd_only():
    return build_dmd_syth_plus_eval(
        db_path=dirs["DMD_only"]["dataset_db_dir"],
        extracted_dir=dirs["DMD_only"]["dataset_extracted_dir"],
        orth_sql=config["sql"]["train"],
        eval_sql=config["sql"]["eval"],
        syth_label="DMD_syth",
        eval_label="DMD_val",
        syth_color="#636EFA",
        eval_color="#E6C243",
        syth_take_n=1000,
    )


def build_chromox():
    samples = load_sql_pipeline(
        db_path=config["paths"]["processed_chromox"] + "/db/dataset_meta.db",
        sql=config["sql"]["chromox_all"],
        parent_dir=config["paths"]["processed_chromox"],
        subsample_n=1000,
    )
    return make_bundle(samples, r"$e^{-}$ Chromox", "#EF553B", name="chromox")


def build_yag():
    samples = load_sql_pipeline(
        db_path=config["paths"]["processed_yag"] + "/db/dataset_meta.db",
        sql=config["sql"]["yag_all"],
        parent_dir=config["paths"]["processed_yag"],
        subsample_n=1000,
    )
    return make_bundle(samples, r"$e^{-}$ Yag", "#00CC96", name="yag")


def build_chromox_laser():
    samples = load_sql_pipeline(
        db_path=config["paths"]["processed_chromox_laser"] + "/db/dataset_meta.db",
        sql=config["sql"]["chromox_laser"],
        parent_dir=config["paths"]["processed_chromox_laser"],
        subsample_n=1000,
    )
    return make_bundle(samples, "laser on Chromox", "#FFA15A", name="chromox_laser")


def build_yag_laser():
    samples = load_sql_pipeline(
        db_path=config["paths"]["processed_yag_laser"] + "/db/dataset_meta.db",
        sql=config["sql"]["yag_laser"],
        parent_dir=config["paths"]["processed_yag_laser"],
        subsample_n=1000,
    )
    return make_bundle(samples, "laser on Yag", "#B6E880", name="yag_laser")


def build_dmd_lab():
    return build_dmd_syth_plus_eval(
        db_path=dirs["DMD_lab"]["dataset_db_dir"],
        extracted_dir=dirs["DMD_lab"]["dataset_extracted_dir"],
        orth_sql=config["sql"]["lab_dmd_orth"],
        eval_sql=config["sql"]["lab_dmd_eval"],
        syth_label="DMD_syth - lab",
        eval_label="DMD_val - lab",
        syth_color="#11B2DF",
        eval_color="#EF1717",
        syth_take_n=1000,
    )


def build_dmd_cockcroft():
    train_samples = load_sql_pipeline(
        db_path=dirs["DMD_cockcroft"]["dataset_db_dir"],
        sql=config["sql"]["cockcroft_dmd_train"],
        parent_dir=dirs["DMD_cockcroft"]["dataset_extracted_dir"],
        subsample_n=5000,
    )
    eval_samples = load_sql_pipeline(
        db_path=dirs["DMD_cockcroft"]["dataset_db_dir"],
        sql=config["sql"]["cockcroft_dmd_eval"],
        parent_dir=dirs["DMD_cockcroft"]["dataset_extracted_dir"],
        subsample_n=1000,
    )

    bundle_train = make_bundle(train_samples, "DMD Cockcroft train", "#11B2DF")
    bundle_eval = make_bundle(eval_samples, "DMD Cockcroft eval", "#EF1717")
    return concat_bundles(bundle_train, bundle_eval, name="dmd_cockcroft")


def build_clear_2022():
    # same source as chromox in your current code
    samples = load_sql_pipeline(
        db_path=config["paths"]["processed_chromox"] + "/db/dataset_meta.db",
        sql=config["sql"]["chromox_all"],
        parent_dir=config["paths"]["processed_chromox"],
        subsample_n=1000,
    )
    return make_bundle(samples, r"$e^{-}$ Chromox", "#EF553B", name="clear_2022")


def build_chromox_line_scan():
    samples = load_sql_pipeline(
        db_path=config["paths"]["processed_chromox"] + "/db/dataset_meta.db",
        sql=config["sql"]["chromox_line_scan"],
        parent_dir=config["paths"]["processed_chromox"],
        subsample_n=5000,
    )
    return make_bundle(samples, r"$e^{-}$ Chromox Line Scan", "#3C6EE4", name="chromox_line_scan")


def build_chromox_random_scan():
    samples = load_sql_pipeline(
        db_path=config["paths"]["processed_chromox"] + "/db/dataset_meta.db",
        sql=config["sql"]["chromox_random_scan"],
        parent_dir=config["paths"]["processed_chromox"],
        subsample_n=3000,
    )
    return make_bundle(samples, r"$e^{-}$ Chromox Random Scan", "#EF553B", name="chromox_random_scan")


# ============================================================
# Select what to build here
# ============================================================
DATASET_BUILDERS = {
    "dmd_only": build_dmd_only,
    "chromox": build_chromox,
    "yag": build_yag,
    "chromox_laser": build_chromox_laser,
    "yag_laser": build_yag_laser,
    "dmd_lab": build_dmd_lab,
    "dmd_cockcroft": build_dmd_cockcroft,
    "clear_2022": build_clear_2022,
    "chromox_line_scan": build_chromox_line_scan,
    "chromox_random_scan": build_chromox_random_scan,
}

selected_names = [
    "clear_2022",
    "yag",
    "chromox_laser",
    "dmd_lab",
    "chromox_line_scan",
    "chromox_random_scan",
]

bundles = {}
for name in selected_names:
    if name not in DATASET_BUILDERS:
        raise KeyError(f"Unknown dataset name: {name}")
    bundles[name] = DATASET_BUILDERS[name]()


# ============================================================
# Merge selected datasets in the given order
# ============================================================
selected_bundles = [bundles[name] for name in selected_names]
merged = concat_bundles(*selected_bundles, name="merged")

all_samples = merged["samples"]   # (2, N_total, C, H, W)
all_labels = merged["labels"]     # len = N_total
all_colors = merged["colors"]     # len = N_total


# Optional:
# downstream single iterable, each item is:
#   sample_pair: (2, C, H, W)
#   label: str
#   color: str
downstream_iterable = list(zip(np.moveaxis(all_samples, 1, 0), all_labels, all_colors))


# ============================================================
# Example visual checks
# ============================================================
if "dmd_lab" in bundles:
    plot_image(bundles["dmd_lab"]["samples"][1][0])

if "chromox_line_scan" in bundles:
    plot_image(bundles["chromox_line_scan"]["samples"][1][99])

if "chromox_random_scan" in bundles:
    plot_image(bundles["chromox_random_scan"]["samples"][1][528])

In [ ]:
# ============================
# Chromox laser data augmentation through linear combination
# ============================






# ============================
# Yag laser data augmentation through linear combination
# ============================







# Latent space distribution

In [ ]:
import numpy as np
import imageio.v2 as imageio
from pathlib import Path
from xflow.utils.visualization import DimReducer, Embedding3DPlot


def embed_images(images, final_method="umap", final_dim=2, random_state=42):
    X = np.asarray(images, dtype=np.float32)
    X = X.reshape(X.shape[0], -1)

    # current behavior: normalize globally
    X /= (X.max() + 1e-12)

    method_norm = "".join(ch for ch in str(final_method).lower() if ch.isalnum())

    if method_norm == "pca":
        pca = DimReducer("pca", n_components=final_dim, random_state=random_state)
        X_final = pca.fit_transform(X)
        return X_final, pca, None

    pca = DimReducer("pca", n_components=min(50, X.shape[1]), random_state=random_state)
    X50 = pca.fit_transform(X)

    if method_norm == "tsne":
        final = DimReducer(
            "tsne",
            n_components=final_dim,
            random_state=random_state,
        )
    elif method_norm == "umap":
        final = DimReducer(
            "umap",
            n_components=final_dim,
            random_state=random_state,
            n_neighbors=10,
            min_dist=0.8,
            metric="euclidean",
        )
    else:
        raise ValueError(f"Unsupported final_method: {final_method}")

    Z = final.fit_transform(X50)
    return Z, pca, final


def prepare_group_data(bundles, selected_names, index):
    """
    Convert bundles -> group_data for selected datasets.

    bundles[name]["samples"]: (2, N, C, H, W)
    """
    group_data = {}

    for name in selected_names:
        if name not in bundles:
            continue

        samples = bundles[name]["samples"][index]   # (N, C, H, W)
        labels = bundles[name]["labels"]
        colors = bundles[name]["colors"]

        group_data[name] = (samples, labels, colors)

    return group_data


def flatten_group_data(group_data, selected_names=None):
    """
    Flatten selected groups into one images / labels / colors set.
    """
    if selected_names is None:
        selected_names = list(group_data.keys())
    else:
        selected_names = [k for k in selected_names if k in group_data]

    if len(selected_names) == 0:
        raise ValueError("No valid dataset selected.")

    images = np.concatenate([group_data[k][0] for k in selected_names], axis=0)
    labels = [label for k in selected_names for label in group_data[k][1]]
    colors = [c for k in selected_names for c in group_data[k][2]]

    return images, labels, colors, selected_names


# ============================================================
# Example usage
# ============================================================
index = 0  # 0: fiber output speckle, 1: original image

selected_names = [
    "chromox_line_scan",
    "chromox_random_scan",
    # "dmd_lab",
    # "clear_2022",
]

group_data = prepare_group_data(bundles, selected_names, index)
images, labels, colors, used_names = flatten_group_data(group_data, selected_names)

for method in ["t-SNE", "UMAP", "PCA"]:  # "t-SNE", "UMAP", "PCA"
    coords, pca_model, final_model = embed_images(
        images,
        final_method=method,
        final_dim=3,
        random_state=42,
    )

    title = f"{method.upper()} projection - {'MMF output speckle' if index == 0 else 'Original images'}"

    plotter = Embedding3DPlot(
        coords=coords,
        labels=labels,
        title=title,
        point_size=2,
        color=colors,
        show_projections=False,
        projection_envelope=False,
        projection_alpha=0.25,
        projection_size_scale=0.7,
        projection_gap_ratio=0.06,
        projection_envelope_alpha=0.18,
    )

    plotter.set_style("darkmode")

    out_dir = Path("results/gif")
    out_dir.mkdir(parents=True, exist_ok=True)

    frames = []
    n_frames = 120
    base_elev, base_azim = 25.0, 40.0
    azim_radius = 25.0 / 2.0
    elev_radius = 10.0 / 2.0
    t = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

    for tt in t:
        azim = base_azim + azim_radius * np.cos(tt)
        elev = base_elev + elev_radius * np.sin(tt)

        frame = plotter.get_matplotlib_frame(
            elev=float(elev),
            azim=float(azim),
            dpi=120,
        )
        frames.append(frame)

    safe_name = "_".join(used_names)
    imageio.mimsave(
        out_dir / f"embedding_orbit_loop_{method}_{safe_name}.gif",
        frames,
        fps=50,
        loop=0,
    )
    plotter.close()

# Fiber coupling rate vs time

In [ ]:
# ============================
# Merge entire CLEAR 2025 dataset in to a single database
# ============================
db_paths = scan_files(dirs["processed_db_dir"], extensions=[".db"], return_type="str")
conn = merge_sqlite_dbs(db_paths, source_column="db_path")
sql = """
SELECT *
FROM mmf_dataset_metadata
"""
tables_df = pd.read_sql_query(sql, conn)
conn.close()
# optional: drop duplicate column names (e.g. both tables have "id", "db_path")
tables_df = tables_df.loc[:, ~tables_df.columns.duplicated()]
print(tables_df.shape)
tables_df.head()

In [ ]:
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

# Minimal inputs from your dataframe
timestamps = tables_df["image_id"].astype(np.int64).tolist()
coupling_ratio = tables_df["coupling_ratio"].astype(float).tolist()
classes = tables_df["image_device"].tolist()

def plot_coupling_time_series(timestamps, coupling_ratio, classes, dot_size=50):
    time_labels = [datetime.fromtimestamp(ts / 1e9).strftime("%Y-%m-%d %H:%M") for ts in timestamps]
    x_indices = list(range(len(coupling_ratio)))

    plt.figure(figsize=(10, 5))

    # 3rd-degree polynomial trend
    z = np.polyfit(x_indices, coupling_ratio, 3)
    p = np.poly1d(z)
    # plt.plot(x_indices, p(x_indices), color="red", dashes=(5, 5), linewidth=1, label="Overall Trend", zorder=3)

    custom_colors = ["C1", "C2", "C0"]
    unique_classes = sorted(set(classes))

    for i, cls in enumerate(unique_classes):
        cls_x = [x_indices[j] for j, c in enumerate(classes) if c == cls]
        cls_y = [coupling_ratio[j] for j, c in enumerate(classes) if c == cls]
        plt.scatter(cls_x, cls_y, label=cls, s=dot_size, color=custom_colors[i % len(custom_colors)], zorder=2)

    step = max(1, len(x_indices) // 10)
    tick_positions = x_indices[::step]
    plt.xticks(tick_positions, [time_labels[i] for i in tick_positions], rotation=15, ha="right")

    plt.xlabel("Time (Sequential)")
    plt.ylabel("Coupling Ratio")
    plt.title("Coupling Ratio vs Time")
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()

plot_coupling_time_series(timestamps, coupling_ratio, classes, dot_size=1)

# Image file size change vs time

In [ ]:
sql = """
SELECT
    d.*,
    c.experiment_description,
    c.image_source,
    c.image_device,
    c.fiber_config,
    c.camera_config,
    c.other_config
FROM mmf_dataset_metadata AS d
LEFT JOIN mmf_experiment_config AS c
  ON c.id = d.config_id
 AND c.db_path = d.db_path;
"""

with sqlite3.connect(str(dirs["merged_db_path"])) as con:
    tables_df = pd.read_sql_query(sql, con)

In [ ]:

from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime


temp = [
    (Path(db).parent.parent / img).as_posix()
    for db, img in zip(tables_df["db_path"], tables_df["image_path"])
]

sizes_kb = [os.path.getsize(path) / 1024 for path in temp]
timestamps = tables_df['image_id'].astype(int).tolist()  # Assuming image_id is a timestamp in seconds
classes = tables_df["image_device"].tolist()  # Assuming "image_device" is the class label

def plot_compressed_time_series(timestamps, sizes_kb, classes, dot_size=50):
    time_labels = [datetime.fromtimestamp(ts / 1e9).strftime('%Y-%m-%d %H:%M') for ts in timestamps]
    x_indices = list(range(len(sizes_kb)))
    
    plt.figure(figsize=(10, 5))
    
    # Fit a 3rd-degree polynomial trend curve across all data points
    z = np.polyfit(x_indices, sizes_kb, 3)
    p = np.poly1d(z)
    plt.plot(x_indices, p(x_indices), color='red', dashes=(5, 5), linewidth=1, label='Overall Trend', zorder=3)
    
    # Define specific colors and sort classes for consistent assignment
    custom_colors = ['C1', 'C2', 'C0']
    unique_classes = sorted(list(set(classes)))
    
    for i, cls in enumerate(unique_classes):
        cls_x = [x_indices[j] for j, c in enumerate(classes) if c == cls]
        cls_y = [sizes_kb[j] for j, c in enumerate(classes) if c == cls]
        
        # Select color, wrapping around if there are more than 3 classes
        color = custom_colors[i % len(custom_colors)]
        
        plt.scatter(cls_x, cls_y, label=cls, s=dot_size, color=color, zorder=2)
    
    # Sample the X-axis ticks
    step = max(1, len(x_indices) // 10)
    tick_positions = x_indices[::step]
    selected_labels = [time_labels[i] for i in tick_positions]
    
    # Apply manual rotation and alignment
    plt.xticks(tick_positions, selected_labels, rotation=15, ha='right')
    
    plt.xlabel('Time (Sequential)')
    plt.ylabel('File Size (KB)')
    plt.title('File Size vs Time')
    
    plt.legend() 
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    
    plt.show()

# Example call:
plot_compressed_time_series(timestamps, sizes_kb, classes, dot_size=1)

# Check Abnormal samples

## Coupling ratio too big or too small samples plot
select some sample where CR smaller than 30 or bigger than 300 for inspection

## Staturation
Select and plot some saturated samples for inspection

# Other interesting samples

# Check if dataset distribution is correct/as expected 

In [ ]:
from xflow.utils.visualization import plot_image, stack_mip

data_type = "chromox_random_scan"  # chromox_random_scan  chromox_line_scan  yag_random_scan  yag_line_scan  
if data_type.startswith("chromox"):
    provider_path = config["paths"]["processed_chromox"]
    provider_db_path = provider_path + "/db/dataset_meta.db"
elif data_type.startswith("yag"):
    provider_path = config["paths"]["processed_yag"]
    provider_db_path = provider_path + "/db/dataset_meta.db"

provider = SqlProvider(
    sources={"connection": provider_db_path, "sql": config["sql"][data_type]},   
    output_config={'list': "image_path"}
)#.subsample(n_samples=10000, seed=config["seed"])

config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": provider_path
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

data_pipeline = PyTorchPipeline(
    provider, 
    transforms[:-1],
).to_numpy() 


mip = stack_mip(data_pipeline[1])
plot_image(mip, title="MIP of {}".format(data_type), cmap="inferno", scale="linear") # inferno, viridis, plasma, magma, gray